# 04 · Entrenamiento y comparación de modelos

Proyecto: **Clasificación anticipada de amenaza meteorológica por lluvias intensas**.

Objetivo del notebook: comparar, bajo las mismas particiones temporales y características, los cinco modelos definidos en la Tarea #4:

1. Random Forest
2. Regresión Logística Multinomial
3. XGBoost
4. SVM con kernel RBF
5. MLP

Además se incluye un `DummyClassifier` únicamente como baseline de referencia.

**Regla metodológica:** el conjunto de prueba temporal y el holdout espacial no se utilizan para ajustar hiperparámetros ni para elegir el modelo ganador.

## 0. Dependencias

In [3]:
# Ejecutar una sola vez si alguna dependencia no está instalada.
# En Colab suele bastar con ejecutar esta celda.
%pip install -q xgboost joblib
%pip install -U scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ------------------- -------------------- 3.9/8.2 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 26.8 MB/s  0:00:00

   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
   ------------- -------------------------- 1/3 [narwhals]
  


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Importaciones y configuración

In [4]:
from pathlib import Path
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_fscore_support,
    recall_score,
)
from sklearn.model_selection import ParameterSampler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Entorno listo.")

Entorno listo.


## 2. Ruta del proyecto

El notebook intenta detectar automáticamente la carpeta del proyecto. Si no la encuentra, cambia `PROJECT_DIR` manualmente.

In [5]:
candidates = [
    Path.cwd(),
    Path.cwd() / "rain-threat-classifier",
    Path("/content/rain-threat-classifier"),
    Path("/content/drive/MyDrive/rain-threat-classifier"),
]

PROJECT_DIR = next(
    (p for p in candidates if (p / "resultados_completo" / "dataset_modelo_mensual.csv").exists()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró el proyecto. Define PROJECT_DIR con la ruta de rain-threat-classifier."
    )

DATA_DIR = PROJECT_DIR / "resultados_completo"
OUTPUT_DIR = PROJECT_DIR / "resultados_modelos"
ARTIFACT_DIR = PROJECT_DIR / "artefactos"

for folder in [OUTPUT_DIR, ARTIFACT_DIR, OUTPUT_DIR / "matrices_confusion", OUTPUT_DIR / "predicciones"]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)

PROJECT_DIR: c:\Users\Kathy\Documents\GitHub\rain-threat-classifier


## 3. Carga del dataset y de las 144 características candidatas

In [6]:
DATASET_PATH = DATA_DIR / "dataset_modelo_mensual.csv"
FEATURES_PATH = DATA_DIR / "columnas_recomendadas_modelo.txt"

df = pd.read_csv(DATASET_PATH)
df["period_start"] = pd.to_datetime(df["period_start"])
df["target_period_start"] = pd.to_datetime(df["target_period_start"])

features = [
    line.strip()
    for line in FEATURES_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("Dataset:", df.shape)
print("Características candidatas:", len(features))
print("Columnas faltantes:", sorted(set(features) - set(df.columns)))

Dataset: (6195, 186)
Características candidatas: 144
Columnas faltantes: []


## 4. Auditoría rápida antes de entrenar

In [7]:
assert df.shape[0] == 6195, f"Se esperaban 6195 filas y llegaron {len(df)}"
assert len(features) == 144, f"Se esperaban 144 características y llegaron {len(features)}"
assert set(features).issubset(df.columns)
assert df[features].isna().sum().sum() == 0, "Hay nulos en las características."
assert df["target_amenaza"].isna().sum() == 0, "Hay targets nulos."
assert set(df["target_amenaza"].unique()) == {"Baja", "Media", "Alta"}

print("Distribución por partición:")
display(df["split"].value_counts().rename("filas").to_frame())

print("Distribución global de clases:")
display(df["target_amenaza"].value_counts().rename("filas").to_frame())

summary = (
    df.groupby("split")
      .agg(
          filas=("split", "size"),
          target_desde=("target_period_start", "min"),
          target_hasta=("target_period_start", "max"),
          zonas=("zone_id", "nunique"),
      )
)
display(summary)

Distribución por partición:


,filas
split,
entrenamiento,3804
holdout_historia,951
validacion_temporal,576
prueba_temporal,576
holdout_espacial,288


Distribución global de clases:


,filas
target_amenaza,
Alta,2143
Baja,2068
Media,1984


,filas,target_desde,target_hasta,zonas
split,,,,
entrenamiento,3804,1991-08-01,2017-12-01,12
holdout_espacial,288,2018-01-01,2025-12-01,3
holdout_historia,951,1991-08-01,2017-12-01,3
prueba_temporal,576,2022-01-01,2025-12-01,12
validacion_temporal,576,2018-01-01,2021-12-01,12


## 5. Separación de particiones

- `entrenamiento`: 12 zonas, objetivos hasta 2017. Se usa para aprender y ajustar hiperparámetros.
- `validacion_temporal`: 2018–2021. Se usa para comparar los cinco modelos ya ajustados.
- `prueba_temporal`: 2022–2025. Se reserva para la evaluación final.
- `holdout_espacial`: Santo Domingo, Nueva Loja y Macas en 2018–2025. Se reserva para evaluar generalización geográfica.
- `holdout_historia`: historia de las zonas reservadas. **No se usa para ajustar el clasificador.**

In [8]:
parts = {name: frame.copy() for name, frame in df.groupby("split")}

train_df = parts["entrenamiento"]
val_df = parts["validacion_temporal"]
test_df = parts["prueba_temporal"]
spatial_df = parts["holdout_espacial"]

X_train = train_df[features]
X_val = val_df[features]
X_test = test_df[features]
X_spatial = spatial_df[features]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df["target_amenaza"])
y_val = label_encoder.transform(val_df["target_amenaza"])
y_test = label_encoder.transform(test_df["target_amenaza"])
y_spatial = label_encoder.transform(spatial_df["target_amenaza"])

CLASS_NAMES = list(label_encoder.classes_)
print("Orden interno de clases:", dict(enumerate(CLASS_NAMES)))
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape, "Spatial:", X_spatial.shape)

Orden interno de clases: {0: 'Alta', 1: 'Baja', 2: 'Media'}
Train: (3804, 144) Val: (576, 144) Test: (576, 144) Spatial: (288, 144)


## 6. Validación cruzada temporal interna

No se usa K-Fold aleatorio. Los folds respetan el orden cronológico y todas las zonas de un mismo periodo objetivo permanecen juntas.

Estos folds solo utilizan `entrenamiento` (hasta 2017). La validación 2018–2021 sigue intacta.

In [9]:
TEMPORAL_FOLDS = [
    ("F1", "2004-12-01", "2005-01-01", "2007-12-01"),
    ("F2", "2007-12-01", "2008-01-01", "2010-12-01"),
    ("F3", "2010-12-01", "2011-01-01", "2013-12-01"),
    ("F4", "2013-12-01", "2014-01-01", "2017-12-01"),
]

def build_temporal_folds(frame):
    folds = []
    description = []
    dates = frame["target_period_start"]

    for name, train_end, val_start, val_end in TEMPORAL_FOLDS:
        train_mask = dates <= pd.Timestamp(train_end)
        val_mask = dates.between(pd.Timestamp(val_start), pd.Timestamp(val_end))

        train_idx = np.flatnonzero(train_mask.to_numpy())
        val_idx = np.flatnonzero(val_mask.to_numpy())
        folds.append((train_idx, val_idx))
        description.append({
            "fold": name,
            "train_filas": len(train_idx),
            "val_filas": len(val_idx),
            "train_hasta": train_end,
            "val_desde": val_start,
            "val_hasta": val_end,
        })

    return folds, pd.DataFrame(description)

temporal_folds, folds_table = build_temporal_folds(train_df)
display(folds_table)

,fold,train_filas,val_filas,train_hasta,val_desde,val_hasta
0,F1,1932,432,2004-12-01,2005-01-01,2007-12-01
1,F2,2364,432,2007-12-01,2008-01-01,2010-12-01
2,F3,2796,432,2010-12-01,2011-01-01,2013-12-01
3,F4,3228,576,2013-12-01,2014-01-01,2017-12-01


## 7. Funciones de evaluación

In [10]:
def calculate_metrics(y_true, y_pred, class_names=CLASS_NAMES):
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(len(class_names)), zero_division=0
    )

    metrics = {
        "macro_f1": macro_f1,
        "balanced_accuracy": balanced_acc,
    }
    for i, class_name in enumerate(class_names):
        key = class_name.lower()
        metrics[f"precision_{key}"] = precision[i]
        metrics[f"recall_{key}"] = recall[i]
        metrics[f"f1_{key}"] = f1[i]
        metrics[f"support_{key}"] = int(support[i])
    return metrics

def show_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASS_NAMES)))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    return fig

def evaluate_model(model, X, y):
    start = time.perf_counter()
    pred = model.predict(X)
    inference_seconds = time.perf_counter() - start
    metrics = calculate_metrics(y, pred)
    metrics["inference_seconds"] = inference_seconds
    return metrics, pred

## 8. Baseline trivial (`DummyClassifier`)

No cuenta entre los cinco modelos. Sirve para saber cuánto mejora el proyecto frente a una estrategia que no aprende patrones meteorológicos.

In [11]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_metrics, dummy_pred = evaluate_model(dummy, X_val, y_val)

print(dummy_metrics)
print(classification_report(y_val, dummy_pred, target_names=CLASS_NAMES, zero_division=0))

{'macro_f1': 0.17309340188517566, 'balanced_accuracy': 0.3333333333333333, 'precision_alta': np.float64(0.3506944444444444), 'recall_alta': np.float64(1.0), 'f1_alta': np.float64(0.519280205655527), 'support_alta': 202, 'precision_baja': np.float64(0.0), 'recall_baja': np.float64(0.0), 'f1_baja': np.float64(0.0), 'support_baja': 202, 'precision_media': np.float64(0.0), 'recall_media': np.float64(0.0), 'f1_media': np.float64(0.0), 'support_media': 172, 'inference_seconds': 0.0005292999994708225}
              precision    recall  f1-score   support

        Alta       0.35      1.00      0.52       202
        Baja       0.00      0.00      0.00       202
       Media       0.00      0.00      0.00       172

    accuracy                           0.35       576
   macro avg       0.12      0.33      0.17       576
weighted avg       0.12      0.35      0.18       576



## 9. Modelos y espacios pequeños de hiperparámetros

Los modelos sensibles a escala (`Logística`, `SVM`, `MLP`) incluyen `StandardScaler` dentro de un `Pipeline`. Random Forest y XGBoost no requieren escalado.

In [12]:
models = {
    "Regresion_Logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=4000, random_state=RANDOM_STATE)),
    ]),
    "Random_Forest": RandomForestClassifier(
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)),
    ]),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            max_iter=600,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=RANDOM_STATE,
        )),
    ]),
}

param_spaces = {
    "Regresion_Logistica": {
        "model__C": [0.01, 0.1, 1.0, 10.0, 50.0],
        "model__class_weight": [None, "balanced"],
    },
    "Random_Forest": {
        "n_estimators": [250, 400, 600],
        "max_depth": [None, 10, 18, 26],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5],
        "class_weight": [None, "balanced"],
    },
    "XGBoost": {
        "n_estimators": [200, 350, 500],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.03, 0.06, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.7, 0.9, 1.0],
    },
    "SVM_RBF": {
        "model__C": [0.1, 1.0, 10.0, 30.0],
        "model__gamma": ["scale", 0.01, 0.001],
        "model__class_weight": [None, "balanced"],
    },
    "MLP": {
        "model__hidden_layer_sizes": [(64,), (128, 64), (128, 64, 32)],
        "model__alpha": [0.0001, 0.001, 0.01],
        "model__learning_rate_init": [0.0005, 0.001, 0.003],
    },
}

print("Modelos definidos:", list(models))

Modelos definidos: ['Regresion_Logistica', 'Random_Forest', 'XGBoost', 'SVM_RBF', 'MLP']


## 10. Búsqueda temporal de hiperparámetros

Se usa una búsqueda aleatoria pequeña para mantener el tiempo de ejecución razonable. Cada configuración se evalúa en los cuatro folds temporales y se selecciona por **Macro F1 promedio**.

In [13]:
def temporal_random_search(
    model_name,
    estimator,
    param_space,
    X,
    y,
    folds,
    n_iter=8,
    random_state=RANDOM_STATE,
):
    sampled_params = list(ParameterSampler(param_space, n_iter=n_iter, random_state=random_state))
    rows = []

    for config_id, params in enumerate(sampled_params, start=1):
        fold_scores = []
        started = time.perf_counter()

        for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):
            candidate = clone(estimator).set_params(**params)
            candidate.fit(X.iloc[train_idx], y[train_idx])
            pred = candidate.predict(X.iloc[val_idx])
            fold_scores.append(f1_score(y[val_idx], pred, average="macro"))

        rows.append({
            "modelo": model_name,
            "config_id": config_id,
            "macro_f1_cv_mean": float(np.mean(fold_scores)),
            "macro_f1_cv_std": float(np.std(fold_scores)),
            "seconds": time.perf_counter() - started,
            "params": params,
        })
        print(
            f"{model_name} | {config_id:02d}/{len(sampled_params)} | "
            f"Macro F1={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}"
        )

    result = pd.DataFrame(rows).sort_values(
        ["macro_f1_cv_mean", "macro_f1_cv_std"],
        ascending=[False, True],
    ).reset_index(drop=True)
    return result

# Para una primera ejecución rápida, 6 configuraciones por modelo es suficiente.
# Si sobra tiempo, subir a 10-15.
N_ITER = 6

search_results = {}
for model_name, estimator in models.items():
    print("\n" + "=" * 80)
    print("AJUSTANDO:", model_name)
    search_results[model_name] = temporal_random_search(
        model_name,
        estimator,
        param_spaces[model_name],
        X_train,
        y_train,
        temporal_folds,
        n_iter=N_ITER,
    )

all_search_results = pd.concat(search_results.values(), ignore_index=True)
all_search_results.to_csv(OUTPUT_DIR / "busqueda_hiperparametros.csv", index=False)
print("Guardado:", OUTPUT_DIR / "busqueda_hiperparametros.csv")


AJUSTANDO: Regresion_Logistica
Regresion_Logistica | 01/6 | Macro F1=0.3248 ± 0.0116
Regresion_Logistica | 02/6 | Macro F1=0.3262 ± 0.0095
Regresion_Logistica | 03/6 | Macro F1=0.3263 ± 0.0086
Regresion_Logistica | 04/6 | Macro F1=0.3181 ± 0.0090
Regresion_Logistica | 05/6 | Macro F1=0.3261 ± 0.0131
Regresion_Logistica | 06/6 | Macro F1=0.3334 ± 0.0141

AJUSTANDO: Random_Forest
Random_Forest | 01/6 | Macro F1=0.3207 ± 0.0186
Random_Forest | 02/6 | Macro F1=0.3265 ± 0.0237
Random_Forest | 03/6 | Macro F1=0.3220 ± 0.0074
Random_Forest | 04/6 | Macro F1=0.3066 ± 0.0146
Random_Forest | 05/6 | Macro F1=0.3175 ± 0.0233
Random_Forest | 06/6 | Macro F1=0.3085 ± 0.0118

AJUSTANDO: XGBoost
XGBoost | 01/6 | Macro F1=0.3315 ± 0.0186
XGBoost | 02/6 | Macro F1=0.3263 ± 0.0106
XGBoost | 03/6 | Macro F1=0.3334 ± 0.0176
XGBoost | 04/6 | Macro F1=0.3350 ± 0.0136
XGBoost | 05/6 | Macro F1=0.3268 ± 0.0185
XGBoost | 06/6 | Macro F1=0.3300 ± 0.0323

AJUSTANDO: SVM_RBF
SVM_RBF | 01/6 | Macro F1=0.3039 ± 0.0

## 11. Mejor configuración por modelo

In [14]:
best_params = {}
for name, result in search_results.items():
    row = result.iloc[0]
    best_params[name] = row["params"]
    print(f"{name}: Macro F1 CV={row['macro_f1_cv_mean']:.4f} | {row['params']}")

with open(OUTPUT_DIR / "mejores_hiperparametros.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False, default=str)

Regresion_Logistica: Macro F1 CV=0.3334 | {'model__class_weight': None, 'model__C': 0.1}
Random_Forest: Macro F1 CV=0.3265 | {'n_estimators': 250, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': 26, 'class_weight': 'balanced'}
XGBoost: Macro F1 CV=0.3350 | {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.03, 'colsample_bytree': 0.9}
SVM_RBF: Macro F1 CV=0.3433 | {'model__gamma': 0.01, 'model__class_weight': 'balanced', 'model__C': 10.0}
MLP: Macro F1 CV=0.3328 | {'model__learning_rate_init': 0.0005, 'model__hidden_layer_sizes': (128, 64), 'model__alpha': 0.01}


## 12. Entrenamiento completo (hasta 2017) y comparación en validación 2018–2021

Esta es la tabla que se utilizará para comparar las cinco familias de modelos. El test temporal y el holdout espacial siguen sin tocarse.

In [15]:
fitted_models = {}
validation_rows = []
validation_predictions = []

# Dummy primero
row = {"modelo": "DummyClassifier", **dummy_metrics, "fit_seconds": np.nan}
validation_rows.append(row)

for name, estimator in models.items():
    model = clone(estimator).set_params(**best_params[name])
    started = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - started

    metrics, pred = evaluate_model(model, X_val, y_val)
    metrics["fit_seconds"] = fit_seconds
    metrics["modelo"] = name
    validation_rows.append(metrics)
    fitted_models[name] = model

    pred_frame = val_df[["zone_id", "ciudad", "period_start", "target_period_start", "target_amenaza"]].copy()
    pred_frame["modelo"] = name
    pred_frame["prediccion"] = label_encoder.inverse_transform(pred)
    validation_predictions.append(pred_frame)

    fig = show_confusion(y_val, pred, f"{name} · Validación temporal 2018–2021")
    fig.savefig(OUTPUT_DIR / "matrices_confusion" / f"{name.lower()}_validacion.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

validation_table = pd.DataFrame(validation_rows)
validation_table = validation_table.sort_values("macro_f1", ascending=False).reset_index(drop=True)
validation_table.to_csv(OUTPUT_DIR / "comparacion_modelos_validacion.csv", index=False)

pd.concat(validation_predictions, ignore_index=True).to_csv(
    OUTPUT_DIR / "predicciones" / "validacion_temporal.csv", index=False
)

display(validation_table[[
    "modelo", "macro_f1", "balanced_accuracy",
    "recall_alta", "f1_alta", "f1_baja", "f1_media",
    "fit_seconds", "inference_seconds"
]])

,modelo,macro_f1,balanced_accuracy,recall_alta,f1_alta,f1_baja,f1_media,fit_seconds,inference_seconds
0,MLP,0.409003,0.409644,0.455446,0.464646,0.437209,0.325153,0.526676,0.003045
1,SVM_RBF,0.373050,0.373839,0.400990,0.406015,0.417062,0.296073,6.920005,0.206342
2,Regresion_Logistica,0.368310,0.368889,0.420792,0.411622,0.399015,0.294294,0.238986,0.003712
3,Random_Forest,0.353780,0.352886,0.371287,0.395778,0.413462,0.252101,3.523640,0.049693
4,XGBoost,0.330018,0.331510,0.257426,0.302326,0.426859,0.260870,1.003535,0.009937
5,DummyClassifier,0.173093,0.333333,1.000000,0.519280,0.000000,0.000000,NaN,0.000529


## 13. Selección del ganador

El criterio principal es Macro F1 en validación temporal. El recall de Alta y la balanced accuracy se revisan como métricas complementarias. La decisión final debe quedar documentada, especialmente si se elige un modelo que no sea el primero por Macro F1.

In [ ]:
candidate_table = validation_table[validation_table["modelo"] != "DummyClassifier"].copy()
winner_name = candidate_table.iloc[0]["modelo"]
print("Ganador provisional por Macro F1:", winner_name)
display(candidate_table[["modelo", "macro_f1", "balanced_accuracy", "recall_alta"]])

## 14. Evaluación final

**Ejecutar esta sección únicamente cuando el grupo haya confirmado el modelo ganador.**

Para la evaluación final, se reentrena el ganador con `entrenamiento + validación temporal` y se evalúa una sola vez en:

- prueba temporal 2022–2025;
- holdout espacial.

No se deben ajustar parámetros después de mirar estas métricas.

In [ ]:
# CONFIRMAR antes de ejecutar. Por defecto toma el ganador por Macro F1.
FINAL_MODEL_NAME = winner_name

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
X_train_val = train_val_df[features]
y_train_val = label_encoder.transform(train_val_df["target_amenaza"])

final_model = clone(models[FINAL_MODEL_NAME]).set_params(**best_params[FINAL_MODEL_NAME])
started = time.perf_counter()
final_model.fit(X_train_val, y_train_val)
final_fit_seconds = time.perf_counter() - started

final_results = {}
for split_name, split_df, X_split, y_split in [
    ("prueba_temporal", test_df, X_test, y_test),
    ("holdout_espacial", spatial_df, X_spatial, y_spatial),
]:
    metrics, pred = evaluate_model(final_model, X_split, y_split)
    final_results[split_name] = metrics

    pred_frame = split_df[["zone_id", "ciudad", "period_start", "target_period_start", "target_amenaza"]].copy()
    pred_frame["prediccion"] = label_encoder.inverse_transform(pred)
    pred_frame.to_csv(OUTPUT_DIR / "predicciones" / f"{split_name}.csv", index=False)

    fig = show_confusion(y_split, pred, f"{FINAL_MODEL_NAME} · {split_name}")
    fig.savefig(OUTPUT_DIR / "matrices_confusion" / f"modelo_final_{split_name}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

print("Modelo final:", FINAL_MODEL_NAME)
print("Tiempo de ajuste final:", round(final_fit_seconds, 3), "s")
display(pd.DataFrame(final_results).T)

## 15. Guardar artefactos finales

In [ ]:
joblib.dump(final_model, ARTIFACT_DIR / "modelo_final.joblib")
joblib.dump(label_encoder, ARTIFACT_DIR / "label_encoder.joblib")

(ARTIFACT_DIR / "caracteristicas_modelo.json").write_text(
    json.dumps(features, indent=2, ensure_ascii=False), encoding="utf-8"
)

metadata = {
    "modelo": FINAL_MODEL_NAME,
    "objetivo": "Clasificar la amenaza meteorológica del mes siguiente",
    "target": "target_amenaza",
    "clases": CLASS_NAMES,
    "n_features": len(features),
    "hiperparametros": best_params[FINAL_MODEL_NAME],
    "train_hasta": "2017-12",
    "validacion": "2018-01 a 2021-12",
    "prueba": "2022-01 a 2025-12",
    "zonas_holdout": sorted(spatial_df["ciudad"].unique().tolist()),
    "metricas_finales": final_results,
}

(ARTIFACT_DIR / "metadata_modelo.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, default=float), encoding="utf-8"
)

print("Artefactos guardados en:", ARTIFACT_DIR)

## 16. Próxima fase

Una vez guardado el modelo final:

1. construir la entrada correspondiente a diciembre de 2025 para estimar enero de 2026;
2. crear una función única de inferencia;
3. desarrollar el prototipo web con selector de zona y periodo;
4. mostrar categoría estimada, probabilidades y antecedentes meteorológicos.

----
## Scripts Adicionales Utilizados
#### // IGNORAR

In [18]:
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Baseline de persistencia:
# predecir que la amenaza del próximo mes será igual a la amenaza del mes actual
y_persistencia_texto = val_df["amenaza_mes"]

# Convertir esas clases al mismo formato numérico de y_val
y_persistencia = label_encoder.transform(y_persistencia_texto)

print(
    "Macro F1:",
    f1_score(y_val, y_persistencia, average="macro")
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_val, y_persistencia)
)

print("\nClassification report:")
print(
    classification_report(
        y_val,
        y_persistencia,
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

print("\nMatriz de confusión:")
print(
    confusion_matrix(
        y_val,
        y_persistencia
    )
)

Macro F1: 0.4092073182130144
Balanced Accuracy: 0.4090682324046358

Classification report:
              precision    recall  f1-score   support

        Alta       0.48      0.48      0.48       202
        Baja       0.45      0.45      0.45       202
       Media       0.29      0.30      0.30       172

    accuracy                           0.41       576
   macro avg       0.41      0.41      0.41       576
weighted avg       0.42      0.41      0.42       576


Matriz de confusión:
[[97 49 56]
 [45 91 66]
 [58 63 51]]


In [19]:
transicion = pd.crosstab(
    val_df["amenaza_mes"],
    val_df["target_amenaza"],
    normalize="index"
)

print(transicion.round(3))

target_amenaza   Alta   Baja  Media
amenaza_mes                        
Alta            0.485  0.225  0.290
Baja            0.241  0.448  0.310
Media           0.324  0.382  0.295
